# The design was chosen for a Gaussian you do not have

You are planning a dose-ranging study. The outcome is a count — conversions,
events, admissions — and you have a response surface fitted to it. You ask
`design_to_identify` for the eight doses that pin down the half-saturation
point best, it hands you a design, and it tells you the study will land
`k_a` to within about 1.3.

Every number in that sentence is right except the last one, and the design is
not the one you wanted. Both errors have the same cause: the information
matrix was built as `J'J / noise_sd²`, which is a statement about a Gaussian
outcome of constant variance, and your outcome is a count whose variance *is*
its mean.

This notebook is about one diagonal matrix that fixes it.

In [ ]:
import numpy as np

from axiom.core import Likelihood, Unsupported
from axiom.design import Weighting, design_to_identify, fisher_information
from axiom.sim import DosePlan, surface_world
from axiom.surface import Design, HillKernel

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, lines

enable();  # every axiom result renders itself from here on

## One weight per row

For any exponential family the log-likelihood's score with respect to the
parameters is

$$\frac{\partial \ell}{\partial\theta}
  = \sum_i \frac{y_i - \mu_i}{\phi\,V(\mu_i)}\,\frac{\partial \mu_i}{\partial\theta}$$

and because $\operatorname{Var}[y_i] = \phi V(\mu_i)$, the Fisher information
is

$$\mathcal{I} = J^\top W J,
  \qquad J = \frac{\partial\mu}{\partial\theta},
  \qquad w_i = \frac{1}{\phi\,V(\mu_i)}.$$

Two things are worth pausing on.

**The link function is not in that formula.** axiom differentiates the *mean*
with respect to $\theta$, not a linear predictor with respect to a coefficient
vector, so whatever link the mean expression contains is already inside $J$ —
computed exactly by `linearize` and `jax.jacfwd`. None of the usual GLM
machinery — inverse links, deviance, iteratively reweighted least squares — is
needed to get a design right.

**The Gaussian case is not a special case.** It is $V \equiv 1$,
$\phi = \sigma^2$, which is $W = I/\sigma^2$ — the `noise_sd` the code always
took. A `Weighting` says which family, and nothing downstream of the matrix
needs to know.

In [ ]:
mu = np.array([1.0, 10.0, 60.0])
rows = []
for w in [
    Weighting(family="normal", scale=2.0),
    Weighting(family="poisson"),
    Weighting(family="gamma", scale=0.5),
    Weighting(family="binomial"),
]:
    at = "n/a" if w.family == "binomial" else np.array2string(
        np.broadcast_to(w.at(mu), mu.shape), precision=4
    )
    rows.append([w.label, at])
table(rows, headers=("weighting", "w at mean = 1, 10, 60"))

The `normal` row is flat: that is what homoscedastic means, and it is why
a Gaussian design can be computed once and used anywhere. Every other row falls
steeply. A Poisson observation where the mean is 60 carries a sixtieth of the
information one at mean 1 does.

`binomial` is absent from that table because it refuses: its mean is a success
probability, and 10 and 60 are not probabilities. A weighting that cannot be
evaluated says so rather than returning a number.

In [ ]:
try:
    Weighting(family="binomial").at(mu)
except ValueError as e:
    print("refused:", e)

## A surface, and two claims about it

A Hill response with a shared intercept, rising from about 1 to about 60 across
the dose range. Nothing about the surface changes in what follows — only what we
assume about the noise around it.

In [ ]:
world = surface_world(
    n_units=1,
    n_periods=24,
    treatments=("a",),
    kernels=HillKernel(reference_dose=50.0, amplitude_scale=10.0),
    doses=DosePlan(scale=50.0),
    intercept="shared",
    truth={"beta_a": 60.0, "alpha": 1.0, "k_a": 50.0, "s_a": 2.0},
    noise_sd=0.5,
    seed=0,
)
surface, theta = world.surface, world.theta

grid = np.array([1.0, 10.0, 25.0, 50.0, 100.0, 200.0, 400.0])
mean_at = np.array(
    [float(np.asarray(surface.forward({"a": np.array([d])}, theta)).ravel()[0]) for d in grid]
)
table(
    [[f"{d:g}", f"{m:.2f}", f"{1 / m:.4f}"] for d, m in zip(grid, mean_at, strict=True)],
    headers=("dose", "mean", "poisson weight"),
)

The weight at the top dose is a sixtieth of the weight at the bottom one.
That is a large enough spread to change a decision, and it does.

In [ ]:
candidates = Design(
    treatments=("a",), points=tuple((float(d),) for d in grid), kind="dose grid"
)
priors = {"alpha": 5.0, "beta_a": 5.0, "k_a": 20.0, "s_a": 1.0}

chosen = {}
for label, kw in [
    ("gaussian", {}),
    ("poisson", {"weighting": Weighting(family="poisson")}),
]:
    out = design_to_identify(
        surface, candidates, theta, 1.0, target="k_a", n=8, prior_sds=priors, seed=0, **kw
    )
    assert not isinstance(out, Unsupported), out
    chosen[label] = out
    doses = sorted(float(p[0]) for p in out.design.points)
    print(f"{label:9s} doses = {[f'{d:g}' for d in doses]}")
    print(f"{'':9s} expected sd(k_a) = {out.expected_sd:.3f}")

Two differences, and the second is the one that would have hurt.

The **designs** are not the same. The Gaussian design spends a row at dose 10
and two at 400; the Poisson design gives both back and buys more replicates at
50 — the half-saturation point, which is where `k_a` lives. It has correctly
worked out that the top of the curve, where a Gaussian sees a perfectly good
observation, is where a count is noisiest.

The **precision claim** is out by a factor of four. The Gaussian arithmetic
promised `k_a` to within 1.3; against the right variance function the same
eight rows deliver about 5.2. A study powered on the first number is a study
that comes back inconclusive and no one can say why.

In [ ]:
fig = lines(
    grid,
    {
        "mean response": mean_at,
        "poisson weight (x100)": 100.0 / mean_at,
    },
    title="Where the information is, for a count outcome",
    subtitle="the response keeps rising; what an observation is worth does not",
    x_title="dose",
    y_title="",
)
caption(
    fig,
    "The two curves run in opposite directions. That is the whole reason the "
    "Poisson design refuses to spend rows at the top of the dose range: the "
    "response is largest there and the information per observation is smallest.",
)

## What a weighting is, and is not

It holds no data. The weights are a function of the mean, so they differ at
every candidate design a search visits — what is constant, and what a result
should carry as provenance, is the *rule*. That is why `FisherInformation`
records a `Weighting` and not a vector.

In [ ]:
fi = fisher_information(surface, dict(world.data), theta, weighting=Weighting(family="poisson"))
assert not isinstance(fi, Unsupported)
print("weighting on the result:", fi.weighting.label)
print("n_observations:        ", fi.n_observations)

# a normal weighting and the noise_sd path are the same number, not merely close
by_sd = fisher_information(surface, dict(world.data), theta, 0.5)
by_weight = fisher_information(
    surface, dict(world.data), theta, weighting=Weighting(family="normal", scale=0.5)
)
print("max |difference|:      ", float(np.max(np.abs(by_sd.as_array() - by_weight.as_array()))))

Information adds over independent rows — that is what makes point exchange
cheap. It adds only over rows in the *same* units, though, and two likelihoods
are not that:

In [ ]:
# by_weight carries its dispersion in W, exactly as fi does, so the two are
# comparable in every respect except the variance function itself.
try:
    _ = fi + by_weight
except ValueError as e:
    print("refused:", e)

print("same weighting adds fine:", np.allclose((fi + fi).as_array(), 2 * fi.as_array()))

A weighting can also be read off a model you have already fitted, which is
the usual way to get one — the dispersion comes from the posterior rather than
from a guess:

In [ ]:
from axiom.core import Data, ModelSpec, Param, Prior, dimensionless

NONE = dimensionless()
scale = Param(name="cv", dimension=NONE, prior=Prior(family="halfnormal", hyper={"sigma": 1.0}))
counts = ModelSpec(
    name="spend",
    mean=Param(name="mu", dimension=NONE, prior=Prior(family="lognormal", hyper={"mu": 0.0, "sigma": 1.0})),
    outcome=Data(name="y", dimension=NONE),
    likelihood=Likelihood(family="gamma", scale="cv"),
    parameters=(scale, Param(name="mu", dimension=NONE, prior=Prior(family="lognormal", hyper={"mu": 0.0, "sigma": 1.0}))),
)
print(Weighting.from_model(counts, {"mu": 3.0, "cv": 0.4}).label)

try:  # a dispersion that was never estimated is not a default
    Weighting.from_model(counts)
except ValueError as e:
    print("refused:", e)

## What this bought you

One diagonal matrix, and with it every design calculation that reads an
information matrix — D-optimality, the identifiability ridge, expected
posterior sd, point exchange — now works for counts, proportions, durations and
positive skewed spend, not only for the Gaussian they were written against.

Two things it did not buy, both worth knowing:

**The design is locally optimal.** The weights depend on the mean, hence on
`theta`, so the design above is best *if* the truth is near where we evaluated
it. That is not new — a Hill kernel already makes `J` depend on `theta`
whatever the likelihood is, and `design.structural` has always said to average
over prior draws. A non-Gaussian family widens that dependence rather than
introducing it.

**A two-arm proportion still needs its own formula.** `difference_se` is
`sd / sqrt(n·a·(1-a))`: one standard deviation shared by both arms. A binary
outcome's variance is a function of its own mean, so the two arms differ unless
the effect is zero — and no per-row weight can rescue a formula that has
already collapsed both arms into one `sd`. Use `proportion_difference_se`, and
notice that the error in the Gaussian version grows with the effect you are
powering for:

In [ ]:
from axiom.design.power import difference_se, power_from_se, proportion_difference_se

n = 400
rows = []
for p_c, p_t in [(0.50, 0.52), (0.40, 0.55), (0.20, 0.60)]:
    exact = proportion_difference_se(p_c, p_t, n)
    pooled = float(np.sqrt(((p_c + p_t) / 2) * (1 - (p_c + p_t) / 2)))
    rows.append([
        f"{p_c:.2f} vs {p_t:.2f}",
        f"{exact:.5f}",
        f"{difference_se(n, pooled):.5f}",
        f"{power_from_se(p_t - p_c, exact).power:.3f}",
    ])
table(rows, headers=("arms", "correct SE", "pooled-sd SE", "power"))

At a two-point difference the pooled shortcut is harmless. At forty points
it is not, and the direction is the bad one: the formula is least trustworthy
exactly where the effect is large enough to be worth studying.